Separable ND convolutionn mother class

---

Kishore Kumar Tarafdar, Date: 29-06-2025


In [1]:
pwd

'/data1/kishoretarafdar/src.port/VolterraMRAsystems.v0/VolterraSys/ndconvolutions/MakingOfQSI3Dand2Dkernels.pynb'

In [2]:
!python --version

Python 3.12.7


        Disable GPU: Force tensorflow to select CPU

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-29 09:12:02.546382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751168522.568180 1249605 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751168522.574947 1249605 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-29 09:12:02.599386: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [5]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

TensorFlow version 2.18.0
Num GPUs Available:  0


2025-06-19 12:41:46.705421: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-06-19 12:41:46.705523: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:137] retrieving CUDA diagnostic information for host: meherangarh
2025-06-19 12:41:46.705534: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:144] hostname: meherangarh
2025-06-19 12:41:46.705868: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:168] libcuda reported version is: 570.148.8
2025-06-19 12:41:46.705921: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:172] kernel reported version is: 570.148.8
2025-06-19 12:41:46.705930: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:259] kernel version seems to match DSO: 570.148.8


0

Select one GPU

        Restrict code to use a particular GPU...

In [4]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [6]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1749154336.629758 3080753 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# Separable conv4d with 2d kernels

        Quadratic NLSI for 2D I/O

    
        Strategy tested with nonseparable conv2d 
        Perfect match with one channel input
        !! Does not match when multiple channel input

        !! Not possible to test the strategy with nonseparable high dimensional convolutions
        (apply update when a libray is located online for nonseparable 4d convolutions)

In [ ]:
import tensorflow as tf
# from tensorflow.keras.layers import Layer

class SeparableConvND(tf.keras.layers.Layer):
    """Separable ND convolution using two 0.5ND kernels mother class

    VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
    Copyright (C) 2025 Kishore Kumar Tarafdar

    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.   
    
    --kkt@29-06-2025"""    
    def __init__(self, filters, kernel=None, kernel_size=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters        
        if kernel is None: 
            self.kernel_size = kernel_size
            self.kernel = kernel 
        else: 
            self.kernel = kernel
            self.kernel_size = tf.shape(kernel)[0]
                    
    def build(self, input_shape):
        self.inchannels = input_shape[-1]
        # self.filters = input_shape[-1]
        ndim = (len(input_shape) - 2)//2  # exclude batch and channel dims
        spatial_shape = input_shape[1:-1]  # [N1, N2, N3, ...]
        
        if self.kernel is None:
            # Experimental pointwise kernel! a nontrainable pointwise convolution with ones to match 
            # input channels to number of output channels (for smooth separable conv with self.kernel)
            # Determine shape for pointwise kernel (1x1x1...x1, in_channels, out_channels)
            pointwise_shape = (1,) * ndim + (self.inchannels, self.filters) 
            # print('ps', pointwise_shape)
            self.pointwise = self.add_weight(
                name='match_inchannels_with_outchannels_number',
                # shape=(1, 1, input_shape[-1], self.filters),
                shape = pointwise_shape,
                initializer='ones',
                trainable=False)
            # Create a 2D kernel that will be applied to both spatial dimensions
            # Shape for separable spatial kernel: (K, K, ..., K, filters, filters)
            kernel_shape = (self.kernel_size,) * ndim + (self.filters, self.filters)
            self.kernel = self.add_weight(
                name='separable_kernel',
                # shape=(self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
                # shape=(self.kernel_size, self.kernel_size, self.filters, self.filters), ## update after pointwise
                shape=kernel_shape,
                initializer='glorot_uniform',
                trainable=True)
    
    def call(self, x):
        pass

    def get_kernel(self):
        """Returns the kernel weights as a numpy array"""
        return self.kernel.numpy()

    # def compute_output_shape(self, input_shape):
    #     return (input_shape[0], input_shape[1], input_shape[2], input_shape[3], input_shape[4], self.filters)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'filters': self.filters,
            'kernel_size': self.kernel_size
        })
        return config

    